# nb_lib_generator
Simulated retail **source system** (pure Python, no Spark). Used by `nb_01_generate_source`.

Each business day it produces the daily extract files described in `docs/design.md` §2:

| Entity | Format | Contents |
|---|---|---|
| customers | CSV | initial load, new sign-ups, profile changes (moves, tier changes) |
| products | JSON lines | initial load, price changes, new products |
| orders | CSV | new orders + status changes (PLACED → SHIPPED → DELIVERED / CANCELLED / RETURNED) |
| order_items | CSV | lines of new orders |

plus the **deliberate data problems** from design §2.3 (duplicates, bad timestamps, orphans,
late headers, stale updates, malformed records), at rate `dq_error_rate`.

**Deterministic**: the same state + day + seed always produces identical files, so a crashed
run can simply be re-run. State (ids, versions, open orders) is kept in a small JSON file.

In [ ]:
import csv
import dataclasses
import datetime as dt
import io
import json
import os
import random

TS_FMT = "%Y-%m-%d %H:%M:%S"

CUSTOMER_COLS = ["customer_id", "first_name", "last_name", "email", "phone", "city", "state",
                 "country", "segment", "loyalty_tier", "signup_date", "updated_at"]
ORDER_COLS = ["order_id", "customer_id", "order_ts", "channel", "payment_method",
              "order_status", "updated_at"]
ITEM_COLS = ["order_line_id", "order_id", "line_no", "product_id", "quantity", "unit_price",
             "discount_pct", "updated_at"]
FILE_FORMATS = {"customers": "csv", "products": "json", "orders": "csv", "order_items": "csv"}

FIRST_NAMES = ["James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda",
               "David", "Elizabeth", "William", "Barbara", "Richard", "Susan", "Joseph", "Jessica",
               "Thomas", "Sarah", "Daniel", "Karen", "Aarav", "Priya", "Wei", "Mei", "Carlos",
               "Sofia", "Omar", "Fatima", "Liam", "Emma"]
LAST_NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis",
              "Rodriguez", "Martinez", "Hernandez", "Lopez", "Wilson", "Anderson", "Thomas",
              "Taylor", "Moore", "Jackson", "Martin", "Lee", "Patel", "Shah", "Chen", "Wang",
              "Kim", "Nguyen", "Khan", "Ali", "Murphy", "Clark"]
CITIES = [("New York", "NY"), ("Los Angeles", "CA"), ("Chicago", "IL"), ("Houston", "TX"),
          ("Phoenix", "AZ"), ("Philadelphia", "PA"), ("San Antonio", "TX"), ("San Diego", "CA"),
          ("Dallas", "TX"), ("Austin", "TX"), ("Seattle", "WA"), ("Denver", "CO"),
          ("Boston", "MA"), ("Atlanta", "GA"), ("Miami", "FL"), ("Portland", "OR"),
          ("Nashville", "TN"), ("Minneapolis", "MN"), ("Charlotte", "NC"), ("Columbus", "OH")]
SEGMENTS = ["CONSUMER", "CORPORATE", "SMALL_BUSINESS"]
TIERS = ["BRONZE", "SILVER", "GOLD", "PLATINUM"]
EMAIL_DOMAINS = ["example.com", "mail.example.org", "shop.example.net"]

CATALOG = {
    "Apparel":     (["T-Shirt", "Jeans", "Jacket", "Dress", "Sweater"], (12, 120)),
    "Footwear":    (["Sneakers", "Boots", "Sandals"], (25, 180)),
    "Electronics": (["Headphones", "Speaker", "Smartwatch", "Charger"], (15, 350)),
    "Home":        (["Mug", "Lamp", "Cushion", "Towel Set"], (8, 90)),
    "Beauty":      (["Shampoo", "Moisturizer", "Perfume"], (6, 95)),
    "Sports":      (["Yoga Mat", "Dumbbell", "Water Bottle", "Backpack"], (10, 120)),
}
BRANDS = ["Northwind", "Contoso", "Fabrikam", "Tailspin", "Litware", "Proseware", "Woodgrove",
          "Adatum"]
COLORS = ["Black", "White", "Blue", "Red", "Green", "Grey", "Beige"]
SIZED_CATEGORIES = {"Apparel": ["XS", "S", "M", "L", "XL"], "Footwear": ["6", "7", "8", "9", "10", "11"]}

CHANNELS = (["STORE", "ONLINE", "MOBILE"], [55, 30, 15])
PAYMENTS = {"STORE": (["CARD", "CASH", "WALLET", "GIFT_CARD"], [60, 25, 10, 5]),
            "ONLINE": (["CARD", "WALLET", "GIFT_CARD"], [70, 25, 5]),
            "MOBILE": (["CARD", "WALLET"], [40, 60])}
ORPHAN_PRODUCT_ID = 999999

## Configuration and state

In [ ]:
@dataclasses.dataclass(frozen=True)
class GeneratorConfig:
    seed: int = 42
    history_days: int = 30          # days generated on the very first run
    initial_customers: int = 2000
    initial_products: int = 200
    orders_per_day: int = 800
    dq_error_rate: float = 0.01     # share of rows with injected problems (0 = clean data)

    def __post_init__(self):
        if self.history_days < 1:
            raise ValueError("history_days must be >= 1")
        if self.initial_customers < 1 or self.initial_products < 1 or self.orders_per_day < 1:
            raise ValueError("initial_customers, initial_products, orders_per_day must be >= 1")
        if not 0 <= self.dq_error_rate <= 0.2:
            raise ValueError("dq_error_rate must be between 0 and 0.2")


def new_state(cfg: GeneratorConfig) -> dict:
    """Empty state: nothing generated yet. Ids start at 1."""
    return {
        "version": 1,
        "seed": cfg.seed,
        "last_date": None,               # last business date generated
        "next_customer_id": 1,
        "next_product_id": 1,
        "next_order_id": 100001,
        "customers": {},                 # "id" -> [profile_version, signup_date]
        "products": {},                  # "id" -> price_version
        "open_orders": [],               # online orders that can still change status
        "deferred_headers": [],          # late-arriving order headers (sent next day)
    }


def _rng(*parts) -> random.Random:
    """Independent, reproducible random generator for any combination of parts."""
    return random.Random("|".join(str(p) for p in parts))


def _ts(day: dt.date, rng: random.Random, start_hour: int = 7, end_hour: int = 22) -> str:
    seconds = rng.randint(start_hour * 3600, end_hour * 3600 - 1)
    return (dt.datetime.combine(day, dt.time()) + dt.timedelta(seconds=seconds)).strftime(TS_FMT)

## Master data: customers and products
Attributes are derived from (seed, id, version), so the state only needs version numbers.

In [ ]:
def customer_row(seed: int, cid: int, version: int, signup_date: str, updated_at: str) -> dict:
    base = _rng(seed, "cust", cid)
    first, last = base.choice(FIRST_NAMES), base.choice(LAST_NAMES)
    city, state = base.choice(CITIES)
    segment = base.choices(SEGMENTS, [70, 20, 10])[0]
    tier = base.choices(TIERS, [55, 30, 12, 3])[0]
    # Each profile version changes one tracked attribute (SCD2 trigger in gold)
    for v in range(1, version + 1):
        change = _rng(seed, "cust", cid, "v", v)
        kind = change.choice(["move", "tier", "segment"])
        if kind == "move":
            city, state = change.choice([c for c in CITIES if c[0] != city])
        elif kind == "tier":
            tier = TIERS[min(TIERS.index(tier) + 1, len(TIERS) - 1)]
        else:
            segment = change.choice([s for s in SEGMENTS if s != segment])
    return {
        "customer_id": cid, "first_name": first, "last_name": last,
        "email": f"{first}.{last}{cid}@{base.choice(EMAIL_DOMAINS)}".lower(),
        "phone": f"+1-{base.randint(200, 989)}-555-{base.randint(0, 9999):04d}",
        "city": city, "state": state, "country": "US",
        "segment": segment, "loyalty_tier": tier,
        "signup_date": signup_date, "updated_at": updated_at,
    }


def product_row(seed: int, pid: int, price_version: int, updated_at: str) -> dict:
    base = _rng(seed, "prod", pid)
    category = base.choice(sorted(CATALOG))
    subcategories, (low, high) = CATALOG[category]
    subcategory, brand, color = base.choice(subcategories), base.choice(BRANDS), base.choice(COLORS)
    size = base.choice(SIZED_CATEGORIES[category]) if category in SIZED_CATEGORIES else None
    base_price = base.uniform(low, high)
    cost_ratio = base.uniform(0.35, 0.65)
    factor = 1.0 if price_version == 0 else _rng(seed, "prod", pid, "p", price_version).uniform(0.85, 1.20)
    price = round(base_price * factor, 2)
    return {
        "product_id": pid, "sku": f"{category[:3].upper()}-{pid:06d}",
        "product_name": f"{brand} {color} {subcategory}",
        "category": category, "subcategory": subcategory, "brand": brand,
        "unit_price": price, "unit_cost": round(base_price * cost_ratio, 2),
        "attributes": {"color": color, "size": size},
        "is_active": True, "updated_at": updated_at,
    }

## One business day

In [ ]:
def _order_count(cfg: GeneratorConfig, day: dt.date, rng: random.Random) -> int:
    weekend = 1.3 if day.weekday() >= 5 else 1.0
    season = {11: 1.15, 12: 1.4}.get(day.month, 1.0)
    return max(1, int(cfg.orders_per_day * weekend * season * rng.uniform(0.9, 1.1)))


def _maybe(rng: random.Random, p: float) -> bool:
    return p > 0 and rng.random() < p


def generate_day(state: dict, day: dt.date, cfg: GeneratorConfig):
    """Generate all extracts for one business day.

    Returns (new_state, files, extra_lines) where files = {entity: [row dicts]} and
    extra_lines = {entity: [raw malformed lines]}. The input state is not modified.
    """
    s = json.loads(json.dumps(state))  # deep copy: keep the function free of side effects
    rng = _rng(cfg.seed, "day", day.isoformat())
    r = cfg.dq_error_rate
    d = day.isoformat()
    initial = s["last_date"] is None
    files = {"customers": [], "products": [], "orders": [], "order_items": []}
    extra = {"customers": [], "products": [], "orders": [], "order_items": []}

    # ---- customers ---------------------------------------------------------------------
    if initial:
        for _ in range(cfg.initial_customers):
            cid = s["next_customer_id"]; s["next_customer_id"] += 1
            signup = (day - dt.timedelta(days=rng.randint(1, 730))).isoformat()
            s["customers"][str(cid)] = [0, signup]
            files["customers"].append(customer_row(cfg.seed, cid, 0, signup, f"{d} 00:00:00"))
    else:
        existing = list(range(1, s["next_customer_id"]))
        for cid in rng.sample(existing, k=min(len(existing), max(1, int(len(existing) * 0.005)))):
            version, signup = s["customers"][str(cid)]
            s["customers"][str(cid)] = [version + 1, signup]
            files["customers"].append(customer_row(cfg.seed, cid, version + 1, signup, _ts(day, rng)))
    for _ in range(max(1, int(cfg.initial_customers * 0.01 * rng.uniform(0.5, 1.5)))):
        cid = s["next_customer_id"]; s["next_customer_id"] += 1
        s["customers"][str(cid)] = [0, d]
        files["customers"].append(customer_row(cfg.seed, cid, 0, d, _ts(day, rng, 0, 7)))

    # ---- products ----------------------------------------------------------------------
    if initial:
        for _ in range(cfg.initial_products):
            pid = s["next_product_id"]; s["next_product_id"] += 1
            s["products"][str(pid)] = 0
            files["products"].append(product_row(cfg.seed, pid, 0, f"{d} 00:00:00"))
    else:
        existing = list(range(1, s["next_product_id"]))
        for pid in rng.sample(existing, k=max(1, int(len(existing) * 0.01))):
            s["products"][str(pid)] += 1
            files["products"].append(product_row(cfg.seed, pid, s["products"][str(pid)], _ts(day, rng, 0, 6)))
        if _maybe(rng, 0.3):
            pid = s["next_product_id"]; s["next_product_id"] += 1
            s["products"][str(pid)] = 0
            files["products"].append(product_row(cfg.seed, pid, 0, _ts(day, rng, 0, 6)))

    # ---- late-arriving headers from yesterday -------------------------------------------
    files["orders"].extend(s["deferred_headers"])
    s["deferred_headers"] = []

    # ---- status changes of open online orders -------------------------------------------
    still_open = []
    for o in s["open_orders"]:
        age = (day - dt.date.fromisoformat(o["placed_date"])).days
        old_status, new_status = o["order_status"], None
        if old_status == "PLACED":
            new_status = "CANCELLED" if _maybe(rng, 0.05) else "SHIPPED"
        elif old_status == "SHIPPED" and _maybe(rng, 0.6):
            new_status = "DELIVERED"
        elif old_status == "DELIVERED" and _maybe(rng, 0.01):
            new_status = "RETURNED"
        if new_status:
            previous_row = {k: o[k] for k in ORDER_COLS}
            o["order_status"], o["updated_at"] = new_status, _ts(day, rng)
            files["orders"].append({k: o[k] for k in ORDER_COLS})
            if _maybe(rng, r):  # stale re-send: an OLDER version arrives after the newer one
                files["orders"].append(previous_row)
        if o["order_status"] in ("PLACED", "SHIPPED") or (o["order_status"] == "DELIVERED" and age <= 14):
            still_open.append(o)
    s["open_orders"] = still_open

    # ---- new orders and lines -----------------------------------------------------------
    product_ids = list(range(1, s["next_product_id"]))
    popularity = [1 / (i ** 0.7) for i in range(1, len(product_ids) + 1)]
    for _ in range(_order_count(cfg, day, rng)):
        oid = s["next_order_id"]; s["next_order_id"] += 1
        channel = rng.choices(*CHANNELS)[0]
        guest = channel == "STORE" and _maybe(rng, 0.2)
        order_ts = _ts(day, rng)
        order = {
            "order_id": oid,
            "customer_id": None if guest else rng.randint(1, s["next_customer_id"] - 1),
            "order_ts": order_ts, "channel": channel,
            "payment_method": rng.choices(*PAYMENTS[channel])[0],
            "order_status": "COMPLETED" if channel == "STORE" else "PLACED",
            "updated_at": order_ts,
        }
        if channel != "STORE":
            s["open_orders"].append({**order, "placed_date": d})
            if _maybe(rng, 0.005):          # header arrives tomorrow, lines arrive today
                s["deferred_headers"].append(order)
            else:
                files["orders"].append(order)
        else:
            files["orders"].append(order)

        for line_no in range(1, rng.choices([1, 2, 3, 4, 5], [50, 25, 13, 8, 4])[0] + 1):
            pid = rng.choices(product_ids, popularity)[0]
            price = product_row(cfg.seed, pid, s["products"][str(pid)], order_ts)["unit_price"]
            files["order_items"].append({
                "order_line_id": f"{oid}-{line_no}", "order_id": oid, "line_no": line_no,
                "product_id": pid, "quantity": rng.choices([1, 2, 3, 4], [70, 18, 8, 4])[0],
                "unit_price": price,
                "discount_pct": rng.choices([0, 0.05, 0.1, 0.15, 0.2, 0.3], [60, 12, 12, 6, 6, 4])[0],
                "updated_at": order_ts,
            })

    _inject_problems(files, extra, rng, r, day)
    s["last_date"] = d
    return s, files, extra


def _inject_problems(files: dict, extra: dict, rng: random.Random, r: float, day: dt.date):
    """Design §2.3: make the data realistically messy. r = 0 leaves the data clean."""
    if r <= 0:
        return
    for row in files["customers"]:
        if _maybe(rng, r * 3):
            row["email"] = f"  {row['email'].upper()} "             # messy text (cleaned in silver)
        elif _maybe(rng, r):
            row["email"] = row["email"].replace("@", "")            # invalid email (warn)
    for row in files["products"]:
        if _maybe(rng, r):
            row["unit_price"] = 0                                   # price_positive (error)
    for row in files["orders"]:
        if _maybe(rng, r * 0.5):
            row["order_ts"] = f"{day.year}-02-30 25:61:00"          # unparseable timestamp (error)
        elif _maybe(rng, r * 0.2):
            row["order_status"] = "PLACD"                           # invalid status (error)
    for row in files["order_items"]:
        if _maybe(rng, r * 0.5):
            row["quantity"] = rng.choice([0, -1, -2])               # quantity_positive (error)
        elif _maybe(rng, r * 0.5):
            row["product_id"] = ORPHAN_PRODUCT_ID                   # product_exists (error, retryable)
        elif _maybe(rng, r * 0.3):
            row["discount_pct"] = 0.75                              # discount_range (warn)
    for entity in ("customers", "orders", "order_items"):           # exact duplicates
        rows = files[entity]
        rows.extend([dict(x) for x in rows if _maybe(rng, r)])
    if _maybe(rng, 0.3):
        extra["order_items"].append("this,is,not,a,valid,order,line,at,all,really")
    if _maybe(rng, 0.3):
        extra["products"].append('{"product_id": 17, "sku": "BROKEN')

## File output and the multi-day driver

In [ ]:
def to_csv(rows: list, columns: list, extra_lines: list = ()) -> str:
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=columns, extrasaction="ignore", lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    for line in extra_lines:
        buf.write(line + "\n")
    return buf.getvalue()


def to_jsonl(rows: list, extra_lines: list = ()) -> str:
    return "".join(json.dumps(r) + "\n" for r in rows) + "".join(l + "\n" for l in extra_lines)


def render_files(day: dt.date, files: dict, extra: dict, landing_dir: str) -> dict:
    """{relative path: file content} for one day, e.g. landing/orders/2026-11-02/orders_20261102.csv"""
    cols = {"customers": CUSTOMER_COLS, "orders": ORDER_COLS, "order_items": ITEM_COLS}
    out = {}
    for entity, rows in files.items():
        ext = FILE_FORMATS[entity]
        path = f"{landing_dir}/{entity}/{day.isoformat()}/{entity}_{day:%Y%m%d}.{ext}"
        out[path] = (to_jsonl(rows, extra[entity]) if ext == "json"
                     else to_csv(rows, cols[entity], extra[entity]))
    return out


def _write_text(root: str, rel_path: str, content: str):
    path = os.path.join(root, rel_path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        f.write(content)
    try:
        os.replace(tmp, path)  # atomic: readers never see a half-written file
    except OSError:            # some mounted file systems cannot rename over an existing file
        if os.path.exists(path):
            os.remove(path)
        os.rename(tmp, path)


def run_generator(root: str, landing_dir: str, state_file: str, run_date: dt.date,
                  cfg: GeneratorConfig, log=None) -> dict:
    """Generate every missing business day up to and including run_date.

    root:       local folder of the lakehouse (Fabric: /lakehouse/default)
    landing_dir, state_file: paths relative to root (e.g. Files/landing, Files/_state/generator_state.json)
    First run generates `history_days` days ending at run_date. Re-running for an already
    generated date does nothing (idempotent). State is saved after every day.
    """
    say = log.info if log else print
    state_path = os.path.join(root, state_file)
    if os.path.exists(state_path):
        with open(state_path, encoding="utf-8") as f:
            state = json.load(f)
        if state.get("seed") != cfg.seed:
            raise ValueError(f"State was created with seed {state.get('seed')}, not {cfg.seed}")
        start = dt.date.fromisoformat(state["last_date"]) + dt.timedelta(days=1)
    else:
        state = new_state(cfg)
        start = run_date - dt.timedelta(days=cfg.history_days - 1)

    summary = {"days": 0, "first_day": None, "last_day": None,
               "rows": {e: 0 for e in FILE_FORMATS}, "files": 0}
    if start > run_date:
        say("source already generated up to %s; nothing to do" % state["last_date"])
        return summary

    day = start
    while day <= run_date:
        state, files, extra = generate_day(state, day, cfg)
        for rel_path, content in render_files(day, files, extra, landing_dir).items():
            _write_text(root, rel_path, content)
            summary["files"] += 1
        _write_text(root, state_file, json.dumps(state))
        for entity, rows in files.items():
            summary["rows"][entity] += len(rows)
        summary["days"] += 1
        summary["first_day"] = summary["first_day"] or day.isoformat()
        summary["last_day"] = day.isoformat()
        say("generated %s: %s" % (day, {e: len(r) for e, r in files.items()}))
        day += dt.timedelta(days=1)
    return summary


print("[nb_lib_generator] loaded")